In [ ]:
# ========== 1. 패키지 ==========
!pip install -q scikit-learn umap-learn

In [ ]:
# ========== 2. import & 모델 ==========
import torch, pandas as pd, numpy as np, re, string, time
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

from huggingface_hub import login
login()  # 실행 후 HF read 토큰 입력

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="cuda",
    attn_implementation="eager",   # attention 추출 위해 필수
)
model.eval()
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
print("model loaded | layers:", model.config.num_hidden_layers)



config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

model loaded | layers: 32


In [ ]:
# ========== 3. 함수 정의 ==========
def build_closedbook_prompt(question):
    msgs = [
        {"role": "system", "content": "Answer the question concisely with only the answer, no explanation."},
        {"role": "user", "content": f"Question: {question}\nAnswer:"},
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def greedy_batch(questions, batch_size=48, max_new=32):
    outs = []
    for i in tqdm(range(0, len(questions), batch_size)):
        batch = [build_closedbook_prompt(q) for q in questions[i:i+batch_size]]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.eos_token_id)
        for j in range(len(batch)):
            new = gen[j][enc["input_ids"].shape[1]:]
            outs.append(tok.decode(new, skip_special_tokens=True).strip())
    return outs

def normalize_answer(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return re.sub(r"\s+", " ", s).strip()

def is_known(generation, candidates):
    gen_norm = normalize_answer(generation)
    if not gen_norm:
        return False
    for c in candidates:
        c_norm = normalize_answer(c)
        if c_norm and c_norm in gen_norm:
            return True
    return False

REFUSAL_PATTERNS = [
    "couldn't find", "could not find", "don't have", "do not have",
    "no information", "not sure", "i'm not aware", "i am not aware",
    "unable to", "cannot find", "can't find", "i don't know", "i do not know",
    "there is no", "not enough information", "insufficient",
]
def detect_refusal(gen):
    g = gen.lower()
    return any(p in g for p in REFUSAL_PATTERNS)

def classify_kstate(gen, candidates):
    if is_known(gen, candidates):
        return "K+"
    if detect_refusal(gen):
        return "K-_refusal"
    return "K-_other"



In [ ]:
# ========== 4. 데이터 로딩 (업로드 후) ==========
from google.colab import files
files.upload()  # conflict_pool_with_candidates.parquet 선택

df_all = pd.read_parquet("conflict_pool_with_candidates.parquet")
# ndarray → list 정규화 (안전)
df_all["answer_candidates"] = df_all["answer_candidates"].apply(
    lambda x: list(x) if isinstance(x, np.ndarray) else x
)
print("로딩 완료:", len(df_all), "rows")

Saving conflict_pool_with_candidates.parquet to conflict_pool_with_candidates (1).parquet
로딩 완료: 32614 rows


In [ ]:
t0 = time.time()
all_gens = greedy_batch(df_all["question"].tolist(), batch_size=48, max_new=32)
print(f"\n생성 완료: {len(all_gens)}개, {(time.time()-t0)/60:.1f}분")

df_all["cb_generation"] = all_gens
df_all["K_state3"] = [classify_kstate(g, c)
                      for g, c in zip(all_gens, df_all["answer_candidates"])]

print("\n=== 전체 K-state 분포 ===")
print(df_all["K_state3"].value_counts())
print("\n=== dataset_id × K_state3 ===")
print(pd.crosstab(df_all["dataset_id"], df_all["K_state3"]))

# 중간 저장 필수
df_all.to_parquet("conflict_pool_kstate.parquet", index=False)
print("\n✓ 저장: conflict_pool_kstate.parquet")

  0%|          | 0/680 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



생성 완료: 32614개, 16.7분

=== 전체 K-state 분포 ===
K_state3
K-_other      18401
K+            10371
K-_refusal     3842
Name: count, dtype: int64

=== dataset_id × K_state3 ===
K_state3      K+  K-_other  K-_refusal
dataset_id                            
clasheval   1801      5663        2715
confiqa_qa  2881      3027          92
dynamicqa   1285      3661         743
mulan       2122      3706         172
nqswap      2282      2344         120

✓ 저장: conflict_pool_kstate.parquet


In [ ]:
# 탐색 풀만 (mulan은 closed_book 검증트랙이라 클러스터링 제외)
explore = df_all[df_all["dataset_id"] != "mulan"].copy()

# K+ / K- 로 단순화 (K-_other + K-_refusal = K-)
explore["K"] = explore["K_state3"].apply(lambda x: "K+" if x == "K+" else "K-")

print("=== 탐색 풀 K 분포 ===")
print(explore["K"].value_counts())

print("\n=== 핵심 셀: fact_nature × K (클러스터링 층화 기준) ===")
print(pd.crosstab(explore["fact_nature"], explore["K"]))

print("\n=== desired_action × K ===")
print(pd.crosstab(explore["desired_action"], explore["K"]))

print("\n=== construction_operator × K ===")
print(pd.crosstab(explore["construction_operator"], explore["K"]))

=== 탐색 풀 K 분포 ===
K
K-    18365
K+     8249
Name: count, dtype: int64

=== 핵심 셀: fact_nature × K (클러스터링 층화 기준) ===
K                   K+     K-
fact_nature                  
disputable         123    571
static            7563  15862
temporal_mutable   563   1932

=== desired_action × K ===
K                     K+    K-
desired_action                
accept              5163  5583
needs_verification  1285  4404
reject              1801  8378

=== construction_operator × K ===
K                        K+    K-
construction_operator            
entity_swap            2579  3367
no_perturbation         722  2472
numeric_perturbation   1504  7475
relation_edit          2881  3119
temporal_replacement    563  1932


In [ ]:
from google.colab import files
files.download("conflict_pool_kstate.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 이어서 진행

In [ ]:
df_all = pd.read_parquet("conflict_pool_kstate.parquet")
df_all["answer_candidates"] = df_all["answer_candidates"].apply(
       lambda x: list(x) if isinstance(x, np.ndarray) else x)
print(len(df_all), df_all["K_state3"].value_counts().to_dict())

32614 {'K-_other': 18401, 'K+': 10371, 'K-_refusal': 3842}


In [ ]:
explore = df_all[df_all["dataset_id"] != "mulan"].copy()
explore["K"] = explore["K_state3"].apply(lambda x: "K+" if x == "K+" else "K-")

# 층화 키: dataset_id × K × desired_action
cell = explore.groupby(["dataset_id", "desired_action", "K"]).size().reset_index(name="n")
print("=== 층화 셀별 크기 ===")
print(cell.to_string(index=False))

print(f"\n최소 셀: {cell['n'].min()}, 셀 개수: {len(cell)}")
print(f"최소 셀 기준 균등 샘플 시 총량: {cell['n'].min() * len(cell)}")

# 셀당 목표 n 후보별 시뮬레이션
for target in [100, 150, 200, 300]:
    usable = cell[cell["n"] >= target]
    print(f"  셀당 {target}개: {len(usable)}/{len(cell)}셀 충족, "
          f"충족셀만 쓰면 총 {len(usable)*target}개")

=== 층화 셀별 크기 ===
dataset_id     desired_action  K    n
 clasheval             reject K+ 1801
 clasheval             reject K- 8378
confiqa_qa             accept K+ 2881
confiqa_qa             accept K- 3119
 dynamicqa needs_verification K+ 1285
 dynamicqa needs_verification K- 4404
    nqswap             accept K+ 2282
    nqswap             accept K- 2464

최소 셀: 1285, 셀 개수: 8
최소 셀 기준 균등 샘플 시 총량: 10280
  셀당 100개: 8/8셀 충족, 충족셀만 쓰면 총 800개
  셀당 150개: 8/8셀 충족, 충족셀만 쓰면 총 1200개
  셀당 200개: 8/8셀 충족, 충족셀만 쓰면 총 1600개
  셀당 300개: 8/8셀 충족, 충족셀만 쓰면 총 2400개


In [ ]:
import numpy as np
np.random.seed(42)

TARGET_PER_CELL = 250

sampled = []
for (did, act, k), grp in explore.groupby(["dataset_id", "desired_action", "K"]):
    n = min(TARGET_PER_CELL, len(grp))
    sampled.append(grp.sample(n=n, random_state=42))
strat = pd.concat(sampled).reset_index(drop=True)

print("=== 층화 샘플 구성 ===")
print(f"총 {len(strat)}개")
print("\n셀별:")
print(strat.groupby(["dataset_id", "desired_action", "K"]).size())

# 재검증 대상 = 이 중 K-_other만 (K+, K-_refusal은 확정)
recheck = strat[strat["K_state3"] == "K-_other"]
print(f"\n재검증 대상 (K-_other): {len(recheck)}개")
print("샘플링 필요 생성 수:", len(recheck), "× 5 =", len(recheck)*5)

# 저장
strat.to_parquet("strat_sample.parquet", index=False)
print("✓ 저장: strat_sample.parquet")

=== 층화 샘플 구성 ===
총 2000개

셀별:
dataset_id  desired_action      K 
clasheval   reject              K+    250
                                K-    250
confiqa_qa  accept              K+    250
                                K-    250
dynamicqa   needs_verification  K+    250
                                K-    250
nqswap      accept              K+    250
                                K-    250
dtype: int64

재검증 대상 (K-_other): 854개
샘플링 필요 생성 수: 854 × 5 = 4270
✓ 저장: strat_sample.parquet


In [ ]:
@torch.no_grad()
def sample_batch(questions, n_samples=5, batch_size=48, max_new=32, temp=0.7):
    """각 질문당 n_samples개 생성. 반환: [[s1..s5], ...]"""
    # 질문을 n_samples배로 복제해서 한 번에 배치 처리
    expanded, owner = [], []
    for qi, q in enumerate(questions):
        for _ in range(n_samples):
            expanded.append(q); owner.append(qi)
    results = [[] for _ in questions]
    for i in tqdm(range(0, len(expanded), batch_size)):
        batch = [build_closedbook_prompt(q) for q in expanded[i:i+batch_size]]
        owners = owner[i:i+batch_size]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=True,
                             temperature=temp, top_p=0.9, pad_token_id=tok.eos_token_id)
        for j in range(len(batch)):
            new = gen[j][enc["input_ids"].shape[1]:]
            results[owners[j]].append(tok.decode(new, skip_special_tokens=True).strip())
    return results

def compute_instability(samples):
    """모달 답에서 벗어나는 비율 (P2 Track C 방식). 정규화 후 최빈값 기준."""
    norm = [normalize_answer(s) for s in samples]
    from collections import Counter
    if not norm: return 1.0
    modal, modal_count = Counter(norm).most_common(1)[0]
    return 1.0 - modal_count / len(norm)

t0 = time.time()
recheck_q = recheck["question"].tolist()
samples = sample_batch(recheck_q, n_samples=5, batch_size=48)
print(f"\n샘플링 완료: {(time.time()-t0)/60:.1f}분")

# 재검증: 5개 중 다수(≥3)가 parametric 포함하면 K+로 승격, 아니면 진짜 K-
recheck = recheck.copy()
recheck["n_known"] = [
    sum(is_known(s, c) for s in samp)
    for samp, c in zip(samples, recheck["answer_candidates"])
]
recheck["instability"] = [compute_instability(s) for s in samples]
recheck["K_final"] = recheck["n_known"].apply(lambda x: "K+" if x >= 3 else "K-")

print("\n=== 재검증 결과 (K-_other 854개) ===")
print("K+로 승격:", (recheck["K_final"]=="K+").sum())
print("K- 확정:", (recheck["K_final"]=="K-").sum())
print("\ninstability 분포:")
print(recheck["instability"].describe())

  0%|          | 0/89 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



샘플링 완료: 2.5분

=== 재검증 결과 (K-_other 854개) ===
K+로 승격: 31
K- 확정: 823

instability 분포:
count    854.000000
mean       0.486651
std        0.278978
min        0.000000
25%        0.200000
50%        0.600000
75%        0.800000
max        0.800000
Name: instability, dtype: float64


In [ ]:
# recheck 결과를 strat에 병합
strat = strat.merge(
    recheck[["sample_id", "n_known", "instability", "K_final"]],
    on="sample_id", how="left"
)

# 최종 K-state 확정
def finalize_k(row):
    if row["K_state3"] == "K+":
        return "K+"                    # greedy에서 이미 확정
    if row["K_state3"] == "K-_refusal":
        return "K-"                    # 명시적 모름 = 확정 K-
    return row["K_final"]              # K-_other는 재검증 결과 사용

strat["K"] = strat.apply(finalize_k, axis=1)

# instability: K+와 K-_refusal은 재검증 안 했으니 NaN → Phase 2에서 필요시 측정
print("=== 최종 K 분포 (층화 2000개) ===")
print(strat["K"].value_counts())
print("\n=== dataset_id × desired_action × K (최종) ===")
print(strat.groupby(["dataset_id","desired_action","K"]).size())

print("\n=== instability 확보 현황 ===")
print("측정됨(K-_other 유래):", strat["instability"].notna().sum())
print("미측정(K+/K-_refusal):", strat["instability"].isna().sum())

strat.to_parquet("strat_final.parquet", index=False)
print("\n✓ 저장: strat_final.parquet — Phase 1 완료")

=== 최종 K 분포 (층화 2000개) ===
K
K+    1031
K-     969
Name: count, dtype: int64

=== dataset_id × desired_action × K (최종) ===
dataset_id  desired_action      K 
clasheval   reject              K+    252
                                K-    248
confiqa_qa  accept              K+    260
                                K-    240
dynamicqa   needs_verification  K+    259
                                K-    241
nqswap      accept              K+    260
                                K-    240
dtype: int64

=== instability 확보 현황 ===
측정됨(K-_other 유래): 854
미측정(K+/K-_refusal): 1146

✓ 저장: strat_final.parquet — Phase 1 완료


## Phase 2

In [ ]:
def build_conflict_prompt_parts(question, context):
    """
    conflict 상황 프롬프트. 구간 경계를 잡기 위해 파트별로 반환.
    반환: (full_prompt_str, 구간 텍스트 dict)
    """
    system = "Answer the question based on the given context."
    ctx_block = f"Context: {context}"
    q_block = f"Question: {question}\nAnswer:"

    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"{ctx_block}\n\n{q_block}"},
    ]
    full = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return full, {"context": ctx_block, "question": q_block}

# 샘플 1개로 구조 확인
ex = strat.iloc[0]
full, parts = build_conflict_prompt_parts(ex["question"], ex["context"])
print("=== 전체 프롬프트 ===")
print(full[:600])
print("...\n")
print("=== 토큰 수 ===")
enc = tok(full, return_tensors="pt")
print("전체 토큰:", enc["input_ids"].shape[1])

# context 블록이 프롬프트 안에서 어느 토큰 범위인지 찾기 (offset mapping)
enc_full = tok(full, return_offsets_mapping=True, add_special_tokens=False)
ctx_start_char = full.find("Context:")
q_start_char = full.find("Question:")
print(f"\nContext 시작 char: {ctx_start_char}, Question 시작 char: {q_start_char}")
print("→ char offset으로 토큰 구간 매핑 가능한지 확인 중")

=== 전체 프롬프트 ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Answer the question based on the given context.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context: Insulin NPH: Drug information -  Insulin NPH: Drug information Formulary drug information for this topic No drug references linked in this topic. Find in topic Formulary Print Share Font Size Small Normal Large Outline Brand Names: US Pharmacologic Category Dosing: Adult Dosing: Kidney Impairment: Adult Dosing: Hepatic Impairment: Adult Dosing: Older Adult Dosing: P
...

=== 토큰 수 ===
전체 토큰: 2445

Context 시작 char: 223, Question 시작 char: 9916
→ char offset으로 토큰 구간 매핑 가능한지 확인 중


In [ ]:
# 층화 2000개의 전체 프롬프트 토큰 길이 측정
lengths = []
for _, ex in tqdm(strat.iterrows(), total=len(strat)):
    full, _ = build_conflict_prompt_parts(ex["question"], ex["context"])
    n_tok = len(tok(full, add_special_tokens=False)["input_ids"])
    lengths.append(n_tok)

strat["prompt_tok_len"] = lengths

print("=== 전체 프롬프트 토큰 길이 분포 ===")
print(strat["prompt_tok_len"].describe())
print("\n=== dataset_id별 길이 (중앙값) ===")
print(strat.groupby("dataset_id")["prompt_tok_len"].agg(["median","min","max"]))
print("\n=== 길이 구간별 개수 ===")
import numpy as np
bins = [0, 256, 512, 1024, 2048, 100000]
print(pd.cut(strat["prompt_tok_len"], bins).value_counts().sort_index())

  0%|          | 0/2000 [00:00<?, ?it/s]

=== 전체 프롬프트 토큰 길이 분포 ===
count     2000.000000
mean       656.737500
std       1397.153911
min         68.000000
25%        145.000000
50%        181.000000
75%        527.500000
max      36465.000000
Name: prompt_tok_len, dtype: float64

=== dataset_id별 길이 (중앙값) ===
            median  min    max
dataset_id                    
clasheval   1599.0  122  36465
confiqa_qa   163.0  115    790
dynamicqa    140.5   68    551
nqswap       205.0   69   2775

=== 길이 구간별 개수 ===
prompt_tok_len
(0, 256]          1327
(256, 512]         166
(512, 1024]        139
(1024, 2048]       230
(2048, 100000]     138
Name: count, dtype: int64


In [ ]:
clash = strat[strat["dataset_id"]=="clasheval"].copy()

positions = []
for _, ex in clash.iterrows():
    ctx = ex["context"]
    ans = str(ex["contextual_answer"])  # perturbed 답이 문맥에 있음
    pos = ctx.find(ans)
    rel = pos / len(ctx) if pos >= 0 and len(ctx) > 0 else -1
    positions.append(rel)

clash["ans_rel_pos"] = positions
print("=== ClashEval 정답 상대 위치 (0=맨앞, 1=맨뒤) ===")
print(clash["ans_rel_pos"].describe())
print("\n정답 못 찾음 (-1):", (clash["ans_rel_pos"]==-1).sum())
print("앞쪽 25% 이내:", (clash["ans_rel_pos"].between(0,0.25)).sum())
print("뒤쪽 75% 이후:", (clash["ans_rel_pos"]>0.75).sum())

# 긴 context 1개 실제로 보기
long_ex = clash.nlargest(1, "prompt_tok_len").iloc[0]
ans = str(long_ex["contextual_answer"])
ctx = long_ex["context"]
p = ctx.find(ans)
print(f"\n=== 최장 샘플 (정답='{ans}', 위치 {p}/{len(ctx)}) ===")
print("정답 주변:", ctx[max(0,p-100):p+100] if p>=0 else "못 찾음")

=== ClashEval 정답 상대 위치 (0=맨앞, 1=맨뒤) ===
count    500.000000
mean       0.204350
std        0.411087
min       -1.000000
25%        0.113675
50%        0.268635
75%        0.402367
max        0.994039
Name: ans_rel_pos, dtype: float64

정답 못 찾음 (-1): 40
앞쪽 25% 이내: 197
뒤쪽 75% 이후: 23

=== 최장 샘플 (정답='Tampa', 위치 154376/158832) ===
정답 주변: e theme park attraction Race Through New York Starring Jimmy Fallon at Universal Studios Florida in Tampa, the Golden Retriever puppies make an appearance, carried inside a basket by Sara's stepdad, G


In [ ]:
def truncate_context(context, answer, tok, max_ctx_tokens=400):
    """정답 중심으로 context를 max_ctx_tokens로 자름. 정답 없으면 앞에서."""
    ctx_ids = tok(context, add_special_tokens=False)["input_ids"]
    if len(ctx_ids) <= max_ctx_tokens:
        return context  # 안 잘라도 됨

    ans = str(answer)
    pos = context.find(ans)
    if pos < 0:
        # 정답 못 찾음 → 앞에서 자름
        return tok.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)

    # 정답의 토큰 위치 추정: char pos 비율로
    ans_tok_pos = int(len(ctx_ids) * pos / len(context))
    half = max_ctx_tokens // 2
    start = max(0, ans_tok_pos - half)
    end = min(len(ctx_ids), start + max_ctx_tokens)
    start = max(0, end - max_ctx_tokens)  # 끝에서 밀리면 보정
    return tok.decode(ctx_ids[start:end], skip_special_tokens=True)

# 적용 + 재측정
strat["context_trunc"] = [
    truncate_context(ex["context"], ex["contextual_answer"], tok, max_ctx_tokens=400)
    for _, ex in tqdm(strat.iterrows(), total=len(strat))
]

# 잘린 후 전체 프롬프트 길이 재확인
new_lens = []
for _, ex in strat.iterrows():
    full, _ = build_conflict_prompt_parts(ex["question"], ex["context_trunc"])
    new_lens.append(len(tok(full, add_special_tokens=False)["input_ids"]))
strat["prompt_tok_len_trunc"] = new_lens

print("=== truncation 후 프롬프트 길이 ===")
print(strat["prompt_tok_len_trunc"].describe())
print("\n=== dataset_id별 (중앙값/최대) ===")
print(strat.groupby("dataset_id")["prompt_tok_len_trunc"].agg(["median","max"]))

# 정답이 잘린 context에 여전히 있는지 검증 (clasheval)
clash_t = strat[strat["dataset_id"]=="clasheval"]
still = sum(str(ex["contextual_answer"]) in ex["context_trunc"] for _, ex in clash_t.iterrows())
print(f"\nClashEval 정답 보존: {still}/{len(clash_t)}")

  0%|          | 0/2000 [00:00<?, ?it/s]

=== truncation 후 프롬프트 길이 ===
count    2000.000000
mean      252.551500
std       144.491388
min        68.000000
25%       145.000000
50%       181.000000
75%       460.000000
max       532.000000
Name: prompt_tok_len_trunc, dtype: float64

=== dataset_id별 (중앙값/최대) ===
            median  max
dataset_id             
clasheval    474.0  532
confiqa_qa   163.0  461
dynamicqa    140.5  459
nqswap       205.0  471

ClashEval 정답 보존: 442/500


In [ ]:
strat.to_parquet("strat_truncated.parquet", index=False)
print("✓ 저장: strat_truncated.parquet")

# attention 추출 전 토큰 구간 경계 잡기 테스트
def get_segment_bounds(question, context, tok):
    """프롬프트를 토큰화하고 context/question 구간의 토큰 인덱스 범위 반환"""
    full, parts = build_conflict_prompt_parts(question, context)
    enc = tok(full, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]

    ctx_char = full.find("Context:")
    q_char = full.find("Question:")
    ans_char = full.rfind("Answer:")

    def char_to_tok(char_pos):
        for i, (s, e) in enumerate(offsets):
            if s <= char_pos < e or s >= char_pos:
                return i
        return len(offsets)

    return {
        "n_tokens": len(offsets),
        "ctx_range": (char_to_tok(ctx_char), char_to_tok(q_char)),
        "q_range": (char_to_tok(q_char), char_to_tok(ans_char)),
        "gen_start": char_to_tok(ans_char),  # answer 이후가 생성 위치
    }

# 3개 샘플로 구간 경계 확인
for i in [0, 500, 1000]:
    ex = strat.iloc[i]
    b = get_segment_bounds(ex["question"], ex["context_trunc"], tok)
    print(f"\n[{ex['dataset_id']}] 총 {b['n_tokens']}토큰")
    print(f"  context: {b['ctx_range']} ({b['ctx_range'][1]-b['ctx_range'][0]} tok)")
    print(f"  question: {b['q_range']} ({b['q_range'][1]-b['q_range'][0]} tok)")
    print(f"  gen_start: {b['gen_start']}")

✓ 저장: strat_truncated.parquet

[clasheval] 총 488토큰
  context: (39, 443) (404 tok)
  question: (443, 481) (38 tok)
  gen_start: 481

[confiqa_qa] 총 152토큰
  context: (39, 134) (95 tok)
  question: (134, 145) (11 tok)
  gen_start: 145

[dynamicqa] 총 128토큰
  context: (39, 101) (62 tok)
  question: (101, 121) (20 tok)
  gen_start: 121


In [ ]:
import numpy as np

@torch.no_grad()
def extract_attention_features(question, context, tok, model):
    """답변 생성 첫 시점의 attention을 context/question 구간별로 집계."""
    b = get_segment_bounds(question, context, tok)
    full, _ = build_conflict_prompt_parts(question, context)
    enc = tok(full, return_tensors="pt", add_special_tokens=False).to("cuda")

    out = model(**enc, output_attentions=True, use_cache=False)
    # out.attentions: tuple(len=32 layers), 각 [1, n_heads, seq, seq]

    ctx_s, ctx_e = b["ctx_range"]
    q_s, q_e = b["q_range"]
    last = enc["input_ids"].shape[1] - 1   # 마지막 토큰(=답변 직전, 다음 토큰 예측 위치)

    feats = {}
    n_layers = len(out.attentions)
    for L in range(n_layers):
        attn = out.attentions[L][0]        # [n_heads, seq, seq]
        attn_last = attn[:, last, :].mean(0)  # head 평균 → [seq]: 마지막 위치가 각 토큰을 본 정도

        ctx_mass = attn_last[ctx_s:ctx_e].sum().item()
        q_mass   = attn_last[q_s:q_e].sum().item()
        ctx_len  = max(ctx_e - ctx_s, 1)
        q_len    = max(q_e - q_s, 1)

        # entropy (context 구간 내 attention 분포의 집중도)
        ctx_a = attn_last[ctx_s:ctx_e]
        ctx_a = ctx_a / (ctx_a.sum() + 1e-9)
        ctx_ent = -(ctx_a * (ctx_a + 1e-9).log()).sum().item()

        feats[f"L{L}_ctx_ratio"]  = ctx_mass                    # context 총 attention
        feats[f"L{L}_q_ratio"]    = q_mass                      # question 총 attention
        feats[f"L{L}_ctx_mean"]   = ctx_mass / ctx_len          # token당 (길이 보정)
        feats[f"L{L}_ctx_ent"]    = ctx_ent                     # 집중도
    return feats

# 파일럿 10개
pilot_feats = []
for _, ex in tqdm(strat.head(10).iterrows(), total=10):
    f = extract_attention_features(ex["question"], ex["context_trunc"], tok, model)
    pilot_feats.append(f)

pf = pd.DataFrame(pilot_feats)
print("feature 차원:", pf.shape[1], "(=32 layer × 4)")
print("\n=== L0, L15, L31의 ctx_ratio (layer별 차이 있는지) ===")
for L in [0, 15, 31]:
    print(f"  L{L}_ctx_ratio: {pf[f'L{L}_ctx_ratio'].mean():.3f} ± {pf[f'L{L}_ctx_ratio'].std():.3f}")
print("\nNaN 있나?:", pf.isna().any().any())
print("메모리 체크 — 샘플당 forward OK")

  0%|          | 0/10 [00:00<?, ?it/s]

feature 차원: 128 (=32 layer × 4)

=== L0, L15, L31의 ctx_ratio (layer별 차이 있는지) ===
  L0_ctx_ratio: 0.437 ± 0.043
  L15_ctx_ratio: 0.068 ± 0.009
  L31_ctx_ratio: 0.081 ± 0.012

NaN 있나?: False
메모리 체크 — 샘플당 forward OK


In [ ]:
import numpy as np

@torch.no_grad()
def extract_features(question, context, tok, model):
    b = get_segment_bounds(question, context, tok)
    full, _ = build_conflict_prompt_parts(question, context)
    enc = tok(full, return_tensors="pt", add_special_tokens=False).to("cuda")

    out = model(**enc, output_attentions=True, output_hidden_states=True, use_cache=False)

    ctx_s, ctx_e = b["ctx_range"]
    q_s, q_e = b["q_range"]
    last = enc["input_ids"].shape[1] - 1

    # --- attention feature (128차원) ---
    attn_feats = {}
    for L in range(len(out.attentions)):
        attn_last = out.attentions[L][0][:, last, :].mean(0)
        ctx_mass = attn_last[ctx_s:ctx_e].sum().item()
        q_mass   = attn_last[q_s:q_e].sum().item()
        ctx_len  = max(ctx_e - ctx_s, 1); q_len = max(q_e - q_s, 1)
        ctx_a = attn_last[ctx_s:ctx_e]; ctx_a = ctx_a/(ctx_a.sum()+1e-9)
        ctx_ent = -(ctx_a*(ctx_a+1e-9).log()).sum().item()
        attn_feats[f"L{L}_ctx_ratio"] = ctx_mass
        attn_feats[f"L{L}_q_ratio"]   = q_mass
        attn_feats[f"L{L}_ctx_mean"]  = ctx_mass/ctx_len
        attn_feats[f"L{L}_ctx_ent"]   = ctx_ent

    # --- hidden state (답변 위치, layer별 4096) ---
    # out.hidden_states: tuple(len=33: embedding + 32 layers), 각 [1, seq, 4096]
    hidden_vec = np.concatenate([
        out.hidden_states[L][0, last, :].float().cpu().numpy()
        for L in range(1, len(out.hidden_states))   # embedding layer(0) 제외
    ])  # → 32*4096 = 131072차원

    return attn_feats, hidden_vec.astype(np.float32)

# 파일럿 5개 (hidden 크기 확인)
attn_list, hid_list = [], []
for _, ex in tqdm(strat.head(5).iterrows(), total=5):
    a, h = extract_features(ex["question"], ex["context_trunc"], tok, model)
    attn_list.append(a); hid_list.append(h)

print("attention feature 차원:", len(attn_list[0]))
print("hidden vector 차원:", hid_list[0].shape, "(=32×4096)")
print("hidden 메모리/샘플:", hid_list[0].nbytes/1e6, "MB")
print("2000개 예상 hidden 크기:", hid_list[0].nbytes*2000/1e9, "GB")

  0%|          | 0/5 [00:00<?, ?it/s]

attention feature 차원: 128
hidden vector 차원: (131072,) (=32×4096)
hidden 메모리/샘플: 0.524288 MB
2000개 예상 hidden 크기: 1.048576 GB


In [ ]:
import time, gc

t0 = time.time()
attn_rows, hidden_mat, sample_ids, fail = [], [], [], []

for i, (_, ex) in enumerate(tqdm(strat.iterrows(), total=len(strat))):
    try:
        a, h = extract_features(ex["question"], ex["context_trunc"], tok, model)
        a["sample_id"] = ex["sample_id"]
        attn_rows.append(a)
        hidden_mat.append(h)
        sample_ids.append(ex["sample_id"])
    except Exception as e:
        fail.append((ex["sample_id"], str(e)[:60]))
    if i % 200 == 0 and i > 0:
        gc.collect(); torch.cuda.empty_cache()

print(f"\n완료: {len(attn_rows)}개, {(time.time()-t0)/60:.1f}분, 실패 {len(fail)}")
if fail: print(fail[:5])

# --- attention: parquet (라벨 병합) ---
attn_df = pd.DataFrame(attn_rows).merge(
    strat[["sample_id","dataset_id","desired_action","fact_nature",
           "construction_operator","K","instability","prompt_tok_len_trunc"]],
    on="sample_id", how="left"
)
attn_df.to_parquet("attention_features.parquet", index=False)

# --- hidden: npy + sample_id 순서 저장 ---
hidden_arr = np.stack(hidden_mat).astype(np.float32)  # [N, 131072]
np.save("hidden_features.npy", hidden_arr)
np.save("hidden_sample_ids.npy", np.array(sample_ids))

print("✓ attention_features.parquet:", attn_df.shape)
print("✓ hidden_features.npy:", hidden_arr.shape)

# sanity: dataset별 L15 ctx_ratio (attention 타입 신호)
print("\n=== dataset_id별 L15_ctx_ratio ===")
print(attn_df.groupby("dataset_id")["L15_ctx_ratio"].agg(["mean","std"]))

  0%|          | 0/2000 [00:00<?, ?it/s]


완료: 2000개, 2.2분, 실패 0
✓ attention_features.parquet: (2000, 136)
✓ hidden_features.npy: (2000, 131072)

=== dataset_id별 L15_ctx_ratio ===
                mean       std
dataset_id                    
clasheval   0.070898  0.020509
confiqa_qa  0.070377  0.014680
dynamicqa   0.072069  0.016502
nqswap      0.075600  0.016943


In [ ]:
# instability 아직 없는 샘플 (K+, K-_refusal 유래)
need = strat[strat["instability"].isna()]
print(f"instability 필요: {len(need)}개 → 샘플링 {len(need)*5}")

t0 = time.time()
need_samples = sample_batch(need["question"].tolist(), n_samples=5, batch_size=48)
print(f"샘플링 완료: {(time.time()-t0)/60:.1f}분")

need = need.copy()
need["instability"] = [compute_instability(s) for s in need_samples]

# strat에 병합 (기존 854개 + 신규 1146개 = 2000 전체)
inst_map = dict(zip(need["sample_id"], need["instability"]))
strat["instability"] = strat.apply(
    lambda r: inst_map.get(r["sample_id"], r["instability"]), axis=1
)

print("\n=== 전체 instability 확보 ===")
print("측정됨:", strat["instability"].notna().sum(), "/ 2000")
print("\n=== K별 instability 평균 (K-가 높아야 P2와 일치) ===")
print(strat.groupby("K")["instability"].agg(["mean","std","count"]))

strat.to_parquet("strat_final_inst.parquet", index=False)
# attention_features에도 갱신된 instability 반영
attn_df = pd.read_parquet("attention_features.parquet")
attn_df["instability"] = attn_df["sample_id"].map(dict(zip(strat["sample_id"], strat["instability"])))
attn_df.to_parquet("attention_features.parquet", index=False)
print("✓ 저장 완료")

instability 필요: 1146개 → 샘플링 5730


  0%|          | 0/120 [00:00<?, ?it/s]

샘플링 완료: 3.1분

=== 전체 instability 확보 ===
측정됨: 2000 / 2000

=== K별 instability 평균 (K-가 높아야 P2와 일치) ===
        mean       std  count
K                            
K+  0.274879  0.295302   1031
K-  0.520743  0.278115    969
✓ 저장 완료
